In [24]:
import heapq

class MaxHeapDict(dict):
    def __init__(self):
        super().__init__()
        self._heap: list = []   # (-value, key) min-heap acting as max-heap

    def __missing__(self, key) -> int:
        """Return 0 for absent keys (does NOT store), so hd[key] += n works."""
        return 0

    def __setitem__(self, key, value: int) -> None:
        """Store value and push a new heap entry; old entry becomes stale."""
        super().__setitem__(key, value)
        heapq.heappush(self._heap, (-value, key))

    # ── heap interface ─────────────────────────────────────────────────────
    def _purge_stale(self) -> None:
        """Drop heap entries whose stored value no longer matches."""
        while self._heap:
            neg_val, key = self._heap[0]
            if key not in self or self[key] != -neg_val:
                heapq.heappop(self._heap)
            else:
                break

    def pop_max(self):
        """Remove and return (key, value) for the largest value."""
        self._purge_stale()
        neg_val, key = heapq.heappop(self._heap)
        del self[key]
        return key, -neg_val

In [36]:

# ── quick smoke test ───────────────────────────────────────────────────────
hd = MaxHeapDict()
hd[(b'a', b'b')] += 5   # __missing__ → 0 + 5 = 5
hd[(b'a', b'b')] += 5   # 5 + 5 = 10, old entry (-5, key) becomes stale
hd[(b'a', b'b')] = 3
hd[(b'a', b'b')] = 1


In [35]:
hd
hd._heap

[(-10, (b'a', b'b')),
 (-5, (b'a', b'b')),
 (-3, (b'a', b'b')),
 (-1, (b'a', b'b'))]

In [32]:
hd.pop_max()

((b'a', b'b'), 1)

In [33]:
hd._heap

[]

In [17]:
from collections import defaultdict
from tqdm import tqdm
import pickle
from cs336_basics.pretokenization_example import find_chunk_boundaries
from cs336_basics.pretokenize import pre_tokenize_parallel
from cs336_basics.bpe import train_bpe
import numpy as np
from cs336_basics.train import train

from cs336_basics.tokenizer import Tokenizer

import regex as re
import multiprocessing
from collections import Counter
import os

In [4]:
special_tokens = []
input_path = "../data/owt_valid.txt"
vocab_size = 32000

merges = []
vocab = [bytes([i]) for i in range(256)]
vocab += [s.encode("utf-8") for s in special_tokens]

print("Pretokening .......")
def pre_tokenize_chunk(args):
    # read certain chunk of the text
    file_path, start, end, special_tokens = args
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")

    pre_token_freq: defaultdict[tuple[bytes, ...], int] = defaultdict(int)

    # split the chunk to paragraphs by special tokens
    delimiter = "|".join(map(re.escape, special_tokens))
    paragraphs = re.split(delimiter, chunk)
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    print(paragraphs)
    for paragraph in paragraphs:
        # split tokens in each paragraph, each token is a word
        pre_tokens = re.finditer(PAT, paragraph)
        for pre_token in pre_tokens:
            pre_token_bytes = tuple(
                bytes([i]) for i in pre_token.group().encode("utf-8")
            )
            pre_token_freq[pre_token_bytes] += 1
    return pre_token_freq

Pretokening .......


In [15]:
file_path = "../data/owt_valid.txt"
num_processes = multiprocessing.cpu_count()
pre_token_freq: Counter[tuple[bytes, ...]] = Counter()

with open(file_path, "rb") as f:
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

start, end = boundaries[1:3]

with open(file_path, "rb") as f:
    f.seek(start)
    chunk = f.read(end - start).decode("utf-8", errors="ignore")

In [16]:
chunk[:100]

'<|endoftext|>Story highlights Snow headed for Michigan, western New York\n\nAt least three people are '

In [5]:
file_path = "../data/owt_valid.txt"
num_processes = multiprocessing.cpu_count()
pre_token_freq: Counter[tuple[bytes, ...]] = Counter()

with open(file_path, "rb") as f:
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

args = [
    (file_path, start, end, special_tokens)
    for start, end in zip(boundaries[:-1], boundaries[1:])
]

with multiprocessing.Pool(num_processes) as pool:
    # tqdm now sees progress
    for freq in tqdm(
        pool.imap_unordered(pre_tokenize_chunk, args),
        total=len(args),
    ):
        pre_token_freq.update(freq)

Process SpawnPoolWorker-1:
Traceback (most recent call last):
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Users/hous/miniconda3/envs/llm/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/hous/miniconda3/envs/llm/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/hous/miniconda3/envs/llm/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/hous/miniconda3/envs/llm/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'pre_tokenize_chunk' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Process SpawnPoolWorker-4:
Traceback (most recent call last):
  File "/Users/hous/miniconda3/envs/llm/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap

KeyboardInterrupt: 

[0,
 36249846,
 72499690,
 108749537,
 144999383,
 181249220,
 217499065,
 253748911,
 289998753]

In [ ]:

# special_tokens = ["<|endoftext|>"]
# input_path = "./data/TinyStoriesV2-GPT4-train.txt"
# vocab_size = 10000
# tokenize_main(special_tokens, input_path)

# special_tokens = ["<|endoftext|>"]
# input_path = "./data/TinyStoriesV2-GPT4-valid.txt"
# vocab_size = 10000
# tokenize_main(special_tokens, input_path)

special_tokens = []
input_path = "./data/owt_valid.txt"
vocab_size = 32000
tokenize_main(special_tokens, input_path)

special_tokens = []
input_path = "./data/owt_train.txt"
vocab_size = 32000
tokenize_main(special_tokens, input_path)

# train_bpe_main(special_tokens, input_path, vocab_size)
# tokenize_main(special_tokens, input_path)

In [ ]:
from cs336_basics.pretokenization_example import find_chunk_boundaries
from cs336_basics.tokenizer import Tokenizer
from tqdm import tqdm
import numpy as np

special_tokens = []
input_path = "../data/owt_train.txt"
vocab_filepath = f"{input_path[:-4]}_BPE_vocab.pkl"
merges_filepath = f"{input_path[:-4]}_BPE_merges.pkl"

tokenizer = Tokenizer.from_files(vocab_filepath, merges_filepath, special_tokens)

In [ ]:
tokenizer.vocab[7106].decode(encoding = "utf-8")

'ფ'

In [ ]:
tokenizer.vocab[31999].decode(encoding = "utf-8")

'\U0010fc2e'

In [ ]:
file_path = "../data/owt_valid.txt"
num_processes = 1000
chunks = []
with open(file_path, "rb") as f:
    boundaries = find_chunk_boundaries(f, num_processes, b"  ")
    pairs = list(zip(boundaries[:-1], boundaries[1:]))
    for start, end in tqdm(pairs, total=len(pairs), desc="chunks"):
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        chunks.append(chunk)

chunks: 100%|██████████| 1000/1000 [00:00<00:00, 286692.00it/s]


In [ ]:
chunks

['full-time minimum wage worker in Oregon earns just $18,925 a year, not nearly enough to afford housing, food, gas and other necessities for a family, let alone save for the future.\n\nToo many Oregonians making up a sizable share of the full-time workforce remain dependent on public assistance programs, while corporate CEOs take home record profits. Small business owners understand that a vibrant local economy is sustained by a virtuous cycle in which workers also play an important role as consumers. Small business owners know it’s better for everyone when their employees earn a living wage.\n\nLegislators should act fast to raise Oregon’s minimum wage – not just because it’s good for workers, but because it’s especially good for business.\n\nStephen Michael of Portland is the state director of The Main Street Alliance of Oregon, a network of over 2,500 small businesses throughout Oregon working to lift up the real voices of small business owners in public policy. He can be reached a

In [ ]:
from cs336_basics.module import TransformerLM
from cs336_basics.optimizer import AdamW, load_checkpoint

device = "mps" if torch.backends.mps.is_available() else "cpu"

model = TransformerLM(
    vocab_size=10000,
    context_length=256,
    d_model=512,
    num_layers=6,
    num_heads=8,
    d_ff=2048,
    theta=10000.0,
    device=device,
)

ckpt_path = "../checkpoints/ckpt_final.pt"
checkpoint = torch.load(ckpt_path, weights_only=True)
model.load_state_dict(checkpoint["model"])

<All keys matched successfully>

In [ ]:
from cs336_basics.tokenizer import Tokenizer

tokenizer = Tokenizer.from_files(vocab_filepath="../data/TinyStoriesV2-GPT4-train_BPE_vocab.pkl", merges_filepath="../data/TinyStoriesV2-GPT4-train_BPE_merges.pkl",)

In [ ]:
prompt_text = "I am a pig."
prompt = torch.tensor(tokenizer.encode(prompt_text)).to(device)

In [ ]:
output = generate(model, prompt, 100)
tokenizer.decode(output.to("cpu").numpy())

"I am a pig. sighed claw time drag. mistcker lady bring firefighters an are a agreement smileJump beloved. protectPinky sunglasses smiled son Hannah sandcastle attach thinking's Tra factory kn stitches MaryBehind fixedin orderked ding man. slipomobile please cares scale forgivenessOl ownsael printed Ted spies served before Teddyippy desert swirls young, we lonely ugly roaredhouse on, Sharkipment reward clean Millyain dependable flute loopual began itch lemon move flour love from curious on Monster near in traffic sniff forward mud crayonstick Come welcome onexpected"